# Validação 29 — desenho determinístico e ficha do artigo

## Objetivo

Validar que metadados científicos têm precedência sobre o Gemini na classificação do desenho e que a ficha do artigo distingue fatos confirmados, sinais não encontrados e itens ainda não avaliados.

## Configuração

O cenário é controlado. Uma tabela é usada porque a ordem de precedência e os estados textuais são mais importantes que magnitudes numéricas.

In [1]:
import sys
from pathlib import Path
from types import SimpleNamespace

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from fatofake.article_profile import build_article_dossier
from fatofake.quality_validation import classify_study_design

## Precedência da classificação

In [2]:
scenarios = [
    ('PubMed prevalece sobre modelo', dict(
        title='Clinical investigation', abstract='',
        publication_types=('Randomized Controlled Trial',),
        llm_context='SYSTEMATIC_REVIEW',
    )),
    ('Identificador de ensaio', dict(
        title='Intervention study', abstract='Registered as NCT12345678.',
    )),
    ('Texto explícito', dict(
        title='Population research', abstract='This cohort study followed adults.',
    )),
    ('Tipo Crossref', dict(
        title='Early results', abstract='', crossref_type='posted-content',
    )),
    ('Fallback do modelo', dict(
        title='Rhodopsin hydration', abstract='Spectroscopy measurements.',
        llm_context='BASIC_SCIENCE',
    )),
]
decisions = [
    {'cenário': name, 'desenho': (result := classify_study_design(**params)).design.value, 'fonte': result.source}
    for name, params in scenarios
]
decisions

[{'cenário': 'PubMed prevalece sobre modelo',
  'desenho': 'RANDOMIZED_CLINICAL_TRIAL',
  'fonte': 'PUBMED_PUBLICATION_TYPE'},
 {'cenário': 'Identificador de ensaio',
  'desenho': 'CLINICAL_TRIAL',
  'fonte': 'TRIAL_IDENTIFIER'},
 {'cenário': 'Texto explícito',
  'desenho': 'OBSERVATIONAL',
  'fonte': 'EXPLICIT_TEXT'},
 {'cenário': 'Tipo Crossref', 'desenho': 'OTHER', 'fonte': 'CROSSREF_TYPE'},
 {'cenário': 'Fallback do modelo',
  'desenho': 'OTHER',
  'fonte': 'GEMINI_FALLBACK'}]

## Ficha controlada

In [3]:
identity = SimpleNamespace(
    status='VERIFIED', reason='DOI e título compatíveis entre PubMed e Crossref.',
    crossref_url='https://doi.org/10.1000/example',
    crossref_authors=('Ana Silva', 'Bruno Souza'),
    crossref_work_type='journal-article', crossref_updates=(),
)
text = 'Methods: We enrolled 240 participants under protocol NCT12345678. Results were reported separately.'
sections = (
    ('Funding', 'Supported by the Example Foundation.'),
    ('Competing interests', 'The authors declare no competing interests.'),
    ('Data availability', 'Data are available in a public repository.'),
)
dossier = build_article_dossier(
    title='Controlled trial', doi='10.1000/example', pmid='12345678',
    authors=('Silva A', 'Souza B'), journal='Example Journal', publication_date='2025',
    publication_types=('Randomized Controlled Trial',), text=text, sections=sections,
    identity=identity, llm_context='SYSTEMATIC_REVIEW', absolute_language=('always works',),
)
{
    'identidade': dossier['identity'],
    'status editorial': dossier['editorial_status'],
    'metodologia': dossier['methodology'],
    'transparência': {key: value['status'] for key, value in dossier['transparency'].items()},
    'resultados versus conclusão': dossier['results_conclusion_consistency'],
}

{'identidade': {'status': 'VERIFIED',
  'title': 'Controlled trial',
  'doi': '10.1000/example',
  'pmid': '12345678',
  'doi_and_title_consistency': 'CONFIRMED',
  'authors_consistency': 'CONSISTENT',
  'authors_explanation': 'PubMed e Crossref possuem autores com nomes compatíveis.',
  'explanation': 'DOI e título compatíveis entre PubMed e Crossref.',
  'source_url': 'https://doi.org/10.1000/example'},
 'status editorial': {'peer_review': 'UNKNOWN',
  'peer_review_explanation': 'Indexação e presença em periódico não confirmam revisão por pares.',
  'retraction': 'NOT_FOUND',
  'retraction_explanation': 'Nenhuma retratação foi localizada; ausência não prova inexistência.',
  'corrections': []},
 'metodologia': {'study_design': 'RANDOMIZED_CLINICAL_TRIAL',
  'classification_source': 'PUBMED_PUBLICATION_TYPE',
  'classification_explanation': 'O PublicationType do PubMed identifica ensaio clínico randomizado.',
  'sample_size': {'status': 'FOUND',
   'value': 240,
   'source': 'EXPLICIT

## Verificações

In [4]:
assert decisions[0]['fonte'] == 'PUBMED_PUBLICATION_TYPE'
assert decisions[0]['desenho'] == 'RANDOMIZED_CLINICAL_TRIAL'
assert decisions[1]['fonte'] == 'TRIAL_IDENTIFIER'
assert decisions[2]['fonte'] == 'EXPLICIT_TEXT'
assert decisions[3]['fonte'] == 'CROSSREF_TYPE'
assert decisions[4]['fonte'] == 'GEMINI_FALLBACK'
assert dossier['identity']['doi_and_title_consistency'] == 'CONFIRMED'
assert dossier['identity']['authors_consistency'] == 'CONSISTENT'
assert dossier['methodology']['classification_source'] == 'PUBMED_PUBLICATION_TYPE'
assert dossier['methodology']['sample_size']['value'] == 240
assert dossier['methodology']['protocol']['identifiers'] == ['NCT12345678']
assert all(value['status'] == 'FOUND' for value in dossier['transparency'].values())
assert dossier['editorial_status']['peer_review'] == 'UNKNOWN'
assert dossier['results_conclusion_consistency']['status'] == 'NOT_EVALUATED'
print('OK: metadados determinísticos prevalecem e ausências não são inventadas.')

OK: metadados determinísticos prevalecem e ausências não são inventadas.


## Próximos passos

A etapa valida a regra de precedência e o contrato da ficha. Ela não confirma revisão por pares nem interpreta conflitos de interesse apenas pela existência de uma seção. A próxima etapa ampliará a busca e aplicará reranking por aderência à alegação.